# CDC Health Dataset

### CDC PLACES 2025 release, using 2023 BRFSS-based estimates

##### Dataset: https://catalog.data.gov/dataset/places-local-data-for-better-health-census-tract-data-2025-release

## Import Libraries

In [1]:
import pandas as pd
import requests

## Load Data

### Each row is one tract and one health measure

In [7]:
bay_area_counties = [
    "06001",  # Alameda
    "06013",  # Contra Costa
    "06041",  # Marin
    "06055",  # Napa
    "06075",  # San Francisco
    "06081",  # San Mateo
    "06085",  # Santa Clara
    "06095",  # Solano
    "06097",  # Sonoma
]

measures = [
    "Obesity among adults",
    "Diagnosed diabetes among adults",
    "Coronary heart disease among adults",
    "Stroke among adults",
    "High blood pressure among adults",
]

api_url = "https://data.cdc.gov/resource/cwsq-ngmh.json"

where = (
    "stateabbr = 'CA' "
    "AND countyfips in (" + ", ".join(f"'{fips}'" for fips in bay_area_counties) + ") "
    "AND measure in (" + ", ".join(f"'{measure}'" for measure in measures) + ")"
)

params = {
    "$select": (
        "year,stateabbr,countyname,countyfips,locationname,measure,"
        "data_value,low_confidence_limit,high_confidence_limit"
    ),
    "$where": where,
    "$limit": 50000,
}

response = requests.get(api_url, params=params, timeout=60)
response.raise_for_status()

health_long = pd.DataFrame(response.json())
health_long.head()
health_long.shape

(8795, 9)

### Reshaping so it is one row per tract

In [10]:
indicator_names = {
    "Obesity among adults": "obesity",
    "Diagnosed diabetes among adults": "diabetes",
    "Coronary heart disease among adults": "coronary_heart_disease",
    "Stroke among adults": "stroke",
    "High blood pressure among adults": "high_blood_pressure",}

health_long["data_value"] = pd.to_numeric(
    health_long["data_value"], errors="coerce")

keys = ["locationname", "countyfips", "countyname"]

def make_wide(value_column, suffix=""):
    wide = health_long.pivot(
        index=keys,
        columns="measure",
        values=value_column
    )
    wide.columns = [
        f"{indicator_names[col]}{suffix}" for col in wide.columns
    ]
    return wide

health_by_tract = (
    make_wide("data_value")
    .join(make_wide("low_confidence_limit", "_lcl"))
    .join(make_wide("high_confidence_limit", "_ucl"))
    .reset_index()
    .rename(columns={"locationname": "GEOID"})
)

health_by_tract["GEOID"] = health_by_tract["GEOID"].astype("string")

health_by_tract.head()

,GEOID,countyfips,countyname,coronary_heart_disease,diabetes,high_blood_pressure,obesity,stroke,coronary_heart_disease_lcl,diabetes_lcl,high_blood_pressure_lcl,obesity_lcl,stroke_lcl,coronary_heart_disease_ucl,diabetes_ucl,high_blood_pressure_ucl,obesity_ucl,stroke_ucl
0,06001400100,06001,Alameda,5.6,9.8,31.7,20.5,2.8,5.0,8.6,28.6,17.1,2.5,6.4,11.4,35.2,24.6,3.1
1,06001400200,06001,Alameda,4.7,7.9,27.1,21.7,2.3,4.1,6.8,24.2,18.1,2.1,5.4,9.2,30.3,26.0,2.7
2,06001400300,06001,Alameda,4.1,7.8,26.3,22.8,2.3,3.6,6.7,23.5,19.1,2.0,4.7,9.0,29.3,26.9,2.6
3,06001400400,06001,Alameda,3.6,6.9,23.8,23.0,2.0,3.2,6.0,21.2,19.2,1.8,4.1,8.0,26.6,27.3,2.2
4,06001400500,06001,Alameda,3.6,7.8,25.1,25.5,2.2,3.2,6.8,22.4,21.3,2.0,4.1,8.9,28.0,30.2,2.5


In [14]:
#reordering columns

id_cols = ["GEOID", "countyfips", "countyname"]

indicator_order = sorted(indicator_names.values())
suffix_order = ["", "_lcl", "_ucl"]  # estimate, lower limit, upper limit

indicator_cols = [
    f"{indicator}{suffix}"
    for indicator in indicator_order
    for suffix in suffix_order
    if f"{indicator}{suffix}" in health_by_tract.columns
]

health_by_tract = health_by_tract[id_cols + indicator_cols]
health_by_tract.head()

,GEOID,countyfips,countyname,coronary_heart_disease,coronary_heart_disease_lcl,coronary_heart_disease_ucl,diabetes,diabetes_lcl,diabetes_ucl,high_blood_pressure,high_blood_pressure_lcl,high_blood_pressure_ucl,obesity,obesity_lcl,obesity_ucl,stroke,stroke_lcl,stroke_ucl
0,06001400100,06001,Alameda,5.6,5.0,6.4,9.8,8.6,11.4,31.7,28.6,35.2,20.5,17.1,24.6,2.8,2.5,3.1
1,06001400200,06001,Alameda,4.7,4.1,5.4,7.9,6.8,9.2,27.1,24.2,30.3,21.7,18.1,26.0,2.3,2.1,2.7
2,06001400300,06001,Alameda,4.1,3.6,4.7,7.8,6.7,9.0,26.3,23.5,29.3,22.8,19.1,26.9,2.3,2.0,2.6
3,06001400400,06001,Alameda,3.6,3.2,4.1,6.9,6.0,8.0,23.8,21.2,26.6,23.0,19.2,27.3,2.0,1.8,2.2
4,06001400500,06001,Alameda,3.6,3.2,4.1,7.8,6.8,8.9,25.1,22.4,28.0,25.5,21.3,30.2,2.2,2.0,2.5


### Saving table as CSV

In [17]:
health_by_tract.to_csv("data/health_by_tract.csv", index=False)